<a href="https://colab.research.google.com/github/iabad5-del/upc-ia/blob/main/SupervisedML_Regressio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MACHINE LEARNING SUPERVISAT: REGRESSIÓ**

## **Preparació de les dades**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_csv('/content/titanic_clean_regressio.csv')

In [ ]:
df

,survived,age,alone,embarked_C,embarked_Q,embarked_S,class_First,class_Second,class_Third,who_child,who_man,who_woman,fare
0,-0.838485,-0.544564,-1.143544,-0.497996,-0.283430,0.612912,-0.610933,-0.515978,0.964724,-0.342751,0.860855,-0.686803,7.2500
1,1.192627,0.573922,-1.143544,2.008048,-0.283430,-1.631555,1.636840,-0.515978,-1.036566,-0.342751,-1.161636,1.456022,71.2833
2,1.192627,-0.264943,0.874475,-0.497996,-0.283430,0.612912,-0.610933,-0.515978,0.964724,-0.342751,-1.161636,1.456022,7.9250
3,1.192627,0.364206,-1.143544,-0.497996,-0.283430,0.612912,1.636840,-0.515978,-1.036566,-0.342751,-1.161636,1.456022,53.1000
4,-0.838485,0.364206,0.874475,-0.497996,-0.283430,0.612912,-0.610933,-0.515978,0.964724,-0.342751,0.860855,-0.686803,8.0500
...,...,...,...,...,...,...,...,...,...,...,...,...,...
775,-0.838485,0.643827,-1.143544,-0.497996,3.528211,-1.631555,-0.610933,-0.515978,0.964724,-0.342751,-1.161636,1.456022,29.1250
776,1.192627,-0.754280,0.874475,-0.497996,-0.283430,0.612912,1.636840,-0.515978,-1.036566,-0.342751,-1.161636,1.456022,30.0000
777,-0.838485,-0.509612,-1.143544,-0.497996,-0.283430,0.612912,-0.610933,-0.515978,0.964724,-0.342751,-1.161636,1.456022,23.4500
778,1.192627,-0.264943,0.874475,2.008048,-0.283430,-1.631555,1.636840,-0.515978,-1.036566,-0.342751,0.860855,-0.686803,30.0000


Aquesta vegada, la variable que intentarem predir serà 'fare', una quantitat numèrica. És per això que estan totes les dades estandarditzades excepte aquesta variable.

In [ ]:
X = df.drop('fare', axis=1)
y = df['fare']

print(f"Variables (X): {X.shape}")
print(f"Target (y): {y.shape}")

Variables (X): (780, 12)
Target (y): (780,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Conjunt de train (shape): {X_train.shape}, {y_train.shape}")
print(f"Conjunt de test (shape): {X_test.shape}, {y_test.shape}")

Conjunt de train (shape): (624, 12), (624,)
Conjunt de test (shape): (156, 12), (156,)


## 1. Linear regression

In [ ]:
pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_regression)),
    ("regressor", LinearRegression())
])

param_grid = {
    "selector__k": [3, 5, 7, 10, 'all']
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("==========================================")
print("Millors paràmetres:")
print(grid_search.best_params_)
print("==========================================")

print("\nMillor RMSE (CV):")
print(-grid_search.best_score_)
print("==========================================")

best_model = grid_search.best_estimator_
selected_features = X_train.columns[
    best_model.named_steps['selector'].get_support()
]

print("Variables seleccionades:")
print(selected_features)

Millors paràmetres:
{'selector__k': 10}

Millor RMSE (CV):
41.29687965821762
Variables seleccionades:
Index(['survived', 'alone', 'embarked_C', 'embarked_Q', 'embarked_S',
       'class_First', 'class_Second', 'class_Third', 'who_man', 'who_woman'],
      dtype='object')


In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nRMSE Test:", rmse)
print("R² Test:", r2)


RMSE Test: 34.067427735238226
R² Test: 0.5155245330012872


## 2. Ridge

In [ ]:
pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_regression)),
    ("regressor", Ridge())
])

param_grid = {
    "selector__k": [3, 5, 7, 10, 'all'],
    "regressor__alpha": [0.01, 0.1, 1, 10, 100]
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("==========================================")
print("Millors paràmetres:")
print(grid_search.best_params_)
print("==========================================")

print("\nMillor RMSE (CV):")
print(-grid_search.best_score_)
print("==========================================")

best_model = grid_search.best_estimator_
selected_features = X_train.columns[
    best_model.named_steps['selector'].get_support()
]

print("Variables seleccionades:")
print(selected_features)

Millors paràmetres:
{'regressor__alpha': 10, 'selector__k': 10}

Millor RMSE (CV):
41.28567497959473
Variables seleccionades:
Index(['survived', 'alone', 'embarked_C', 'embarked_Q', 'embarked_S',
       'class_First', 'class_Second', 'class_Third', 'who_man', 'who_woman'],
      dtype='object')


In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nRMSE Test:", rmse)
print("R² Test:", r2)


RMSE Test: 34.095896605392035
R² Test: 0.5147144787644826


## 3. Lasso

In [ ]:
pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_regression)),
    ("regressor", Lasso())
])

param_grid = {
    "selector__k": [3, 5, 7, 10, 'all'],
    "regressor__alpha": [0.01, 0.1, 1, 10, 100]
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("==========================================")
print("Millors paràmetres:")
print(grid_search.best_params_)
print("==========================================")

print("\nMillor RMSE (CV):")
print(-grid_search.best_score_)
print("==========================================")

best_model = grid_search.best_estimator_
selected_features = X_train.columns[
    best_model.named_steps['selector'].get_support()
]

print("Variables seleccionades:")
print(selected_features)

Millors paràmetres:
{'regressor__alpha': 0.1, 'selector__k': 10}

Millor RMSE (CV):
41.28887715968297
Variables seleccionades:
Index(['survived', 'alone', 'embarked_C', 'embarked_Q', 'embarked_S',
       'class_First', 'class_Second', 'class_Third', 'who_man', 'who_woman'],
      dtype='object')


In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nRMSE Test:", rmse)
print("R² Test:", r2)


RMSE Test: 34.06627741335396
R² Test: 0.5155572500779894


## 4. SVR

In [ ]:
pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_regression)),
    ("regressor", SVR())
])

param_grid = {
    "selector__k": [3, 5, 7, 10, 'all'],
    "regressor__C": [0.01, 0.1, 1, 10, 100],
    "regressor__kernel": ['linear', 'poly', 'rbf', 'sigmoid'],
    "regressor__epsilon": [0.01, 0.1, 1, 10, 100],
    "regressor__degree": [2,3,4]
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("==========================================")
print("Millors paràmetres:")
print(grid_search.best_params_)
print("==========================================")

print("\nMillor RMSE (CV):")
print(-grid_search.best_score_)
print("==========================================")

best_model = grid_search.best_estimator_
selected_features = X_train.columns[
    best_model.named_steps['selector'].get_support()
]

print("Variables seleccionades:")
print(selected_features)

Millors paràmetres:
{'regressor__C': 100, 'regressor__degree': 3, 'regressor__epsilon': 10, 'regressor__kernel': 'poly', 'selector__k': 10}

Millor RMSE (CV):
40.66652815452434
Variables seleccionades:
Index(['survived', 'alone', 'embarked_C', 'embarked_Q', 'embarked_S',
       'class_First', 'class_Second', 'class_Third', 'who_man', 'who_woman'],
      dtype='object')


In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nRMSE Test:", rmse)
print("R² Test:", r2)


RMSE Test: 34.40075773388941
R² Test: 0.5059975295554628


In [ ]:
y_test

,fare
595,8.0500
587,7.7750
543,7.0500
644,26.0000
487,26.0000
...,...
351,7.9250
79,7.7875
148,7.7333
333,14.4542


In [ ]:
y_pred

array([ 16.23751891,  16.23751891,  16.23751891,  30.99987284,
        35.99982329,  16.23751891,  20.49977652,  27.9051136 ,
        29.00009898,   3.49740546,  16.23751891,  20.49977652,
        34.14990057,  16.23751891,  16.23751891,  16.23751891,
        69.65008833,  18.34651866,  88.26696412,  16.23751891,
        16.23751891,  17.79572501,  17.79572501,  17.79572501,
        36.28747807,  61.99977467, 122.25872122,   3.49740546,
       122.25872122,  15.43503052,  29.00009898,  16.23751891,
        30.99987284, 122.25872122,  20.50005117,  20.50005117,
        10.75893864,  20.49977652,  34.00037368,  16.23751891,
        17.24982111,  87.95800579,  15.43503052,  16.23751891,
        21.27498195,  27.9051136 ,  15.43503052,  29.00019618,
        16.23751891,  69.65008833,  35.99982329,  16.23751891,
        34.14990057,  67.28739076,  30.99987284,  16.23751891,
        16.23751891,  16.23751891,  29.00019618,  35.99982329,
        86.29166412,  44.65454606,  17.24982111,  20.50